<a target="_blank" href="https://colab.research.google.com/github/genomicsxai/alphagenome-pytorch/blob/main/examples/notebooks/finetune_atac.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# AlphaGenome-PyTorch Finetuning Tutorial

This notebook demonstrates how to finetune AlphaGenome on custom ATAC-seq tracks using linear probing (frozen trunk, trainable head).

## Prerequisites

You will need:
- **Pretrained weights**: AlphaGenome PyTorch checkpoint (`.pt` file)
- **Reference genome**: FASTA file (e.g., `hg38.fa`) with `.fai` index
- **BigWig files**: Signal tracks for your samples (one per track/cell type)
- **BED files**: Training and validation genomic regions

## Overview

1. Install dependencies
2. Configure paths and hyperparameters
3. Compute track statistics
4. Create datasets and data loaders
5. Initialize model with frozen trunk and new head
6. Train the model
7. Save checkpoint
8. Run inference
9. Visualize predictions

## 1. Install dependencies

In [ ]:
# %pip install alphagenome-pytorch pyBigWig pyfaidx matplotlib

## 2. Imports

In [ ]:
import numpy as np

import torch
from torch.utils.data import DataLoader
from pathlib import Path

from alphagenome_pytorch import AlphaGenome
from alphagenome_pytorch.extensions.finetuning import (
    GenomicDataset,
    compute_track_means,
    collate_genomic,
    TransferConfig,
    save_checkpoint,
)
from alphagenome_pytorch.extensions.finetuning.transfer import (
    load_trunk,
    prepare_for_transfer,
    remove_all_heads,
    add_head,
)
from alphagenome_pytorch.extensions.finetuning.heads import create_finetuning_head
from alphagenome_pytorch.extensions.finetuning.training import (
    train_epoch,
    validate,
    create_lr_scheduler,
)

## 3. Configuration

Update these paths to point to your data files.

In [ ]:
# =============================================================================
# PATHS - UPDATE THESE FOR YOUR DATA
# =============================================================================

PRETRAINED_WEIGHTS = '/path/to/alphagenome_pytorch.pt'  # Pretrained model weights
GENOME_FASTA = '/path/to/hg38.fa'                       # Reference genome with .fai index

# BigWig signal files (one per track/sample)
BIGWIG_FILES = [
    '/path/to/sample1.bw',
    '/path/to/sample2.bw',
    # Add more samples as needed
]

# BED files with genomic regions (chrom, start, end)
TRAIN_BED = '/path/to/train_regions.bed'
VAL_BED = '/path/to/val_regions.bed'

# Output directory
OUTPUT_DIR = Path('./finetuning_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# HYPERPARAMETERS
# =============================================================================

SEQUENCE_LENGTH = 131_072    # Input sequence length (128KB)
RESOLUTIONS = (1, 128)       # Output resolutions (1bp and 128bp)
BATCH_SIZE = 2               # Batch size (reduce if OOM)
LEARNING_RATE = 1e-3         # Learning rate for linear probing
NUM_EPOCHS = 10              # Number of training epochs
WARMUP_STEPS = 100           # LR warmup steps
POSITIONAL_WEIGHT = 5.0      # Weight for positional loss component
LOG_EVERY = 50               # Log frequency (batches)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

## 4. Compute Track Statistics

Compute the mean signal per track from training data. This is used to properly scale model outputs.

In [ ]:
print("Computing track means from training data...")
track_means = compute_track_means(
    bigwig_files=BIGWIG_FILES,
    bed_file=TRAIN_BED,
    sequence_length=SEQUENCE_LENGTH,
    max_samples=1000,  # Use subset for speed; set to None for all samples
)
print(f"Track means shape: {track_means.shape}")
print(f"Track means: {track_means}")

## 5. Create Datasets

Load genomic sequences and signals using `GenomicDataset`.

In [ ]:
print("Creating datasets...")

# Training dataset with caching for speed
train_dataset = GenomicDataset(
    genome_fasta=GENOME_FASTA,
    bigwig_files=BIGWIG_FILES,
    bed_file=TRAIN_BED,
    resolutions=RESOLUTIONS,
    sequence_length=SEQUENCE_LENGTH,
    cache_genome=True,   # Prefetch genome (~12GB RAM for hg38)
    cache_signals=True,  # Prefetch BigWig signals
)

# Validation dataset
val_dataset = GenomicDataset(
    genome_fasta=GENOME_FASTA,
    bigwig_files=BIGWIG_FILES,
    bed_file=VAL_BED,
    resolutions=RESOLUTIONS,
    sequence_length=SEQUENCE_LENGTH,
    cache_genome=True,
    cache_signals=True,
)

print(f"Train samples: {len(train_dataset):,}")
print(f"Val samples: {len(val_dataset):,}")

In [ ]:
# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_genomic,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_genomic,
)

print(f"Train batches: {len(train_loader):,}")
print(f"Val batches: {len(val_loader):,}")

## 6. Initialize Model (Linear Probing)

Load pretrained trunk, remove original heads, and add a new ATAC head for finetuning.

In [ ]:
# Initialize model with gradient checkpointing for memory efficiency
print("Loading pretrained model...")
model = AlphaGenome()

# Load pretrained trunk weights (excluding heads)
model = load_trunk(model, PRETRAINED_WEIGHTS, exclude_heads=True)

# Freeze trunk parameters (linear probing)
for param in model.parameters():
    param.requires_grad = False

# Remove all original heads
model = remove_all_heads(model)

# Create new ATAC head for our tracks
n_tracks = len(BIGWIG_FILES)
head = create_finetuning_head(
    assay_type='atac',
    n_tracks=n_tracks,
    resolutions=list(RESOLUTIONS),
    num_organisms=1,
    track_means=track_means,
)

# Register head on model
add_head(model, 'my_atac', head)

# Move to device
model = model.to(DEVICE)

# Get reference to head on device
head = model.heads['my_atac']

print(f"Created ATAC head with {n_tracks} tracks at resolutions {RESOLUTIONS}")

In [ ]:
# Count trainable parameters
trainable_params = [p for p in model.parameters() if p.requires_grad]
n_trainable = sum(p.numel() for p in trainable_params)
n_total = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.2f}%)")

## 7. Setup Optimizer and Scheduler

In [ ]:
# Optimizer - only train head parameters
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=0.1,
)

# Learning rate scheduler with warmup + cosine decay
total_steps = NUM_EPOCHS * len(train_loader)
scheduler = create_lr_scheduler(
    optimizer,
    warmup_steps=WARMUP_STEPS,
    total_steps=total_steps,
    schedule='cosine',
)

print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps: {WARMUP_STEPS}")

## 8. Training Loop

In [ ]:
# Resolution weights (equal weight for all resolutions)
resolution_weights = {res: 1.0 for res in RESOLUTIONS}

# Track names for checkpoint metadata
track_names = [Path(bw).stem for bw in BIGWIG_FILES]

# Training loop
best_val_loss = float('inf')

print("\n" + "=" * 60)
print("Starting training")
print("=" * 60 + "\n")

for epoch in range(1, NUM_EPOCHS + 1):
    # Train for one epoch
    train_loss = train_epoch(
        model=model,
        head=head,
        train_loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        device=DEVICE,
        resolution_weights=resolution_weights,
        positional_weight=POSITIONAL_WEIGHT,
        epoch=epoch,
        log_every=LOG_EVERY,
        use_amp=True,
    )
    
    # Validate
    val_loss = validate(
        model=model,
        head=head,
        val_loader=val_loader,
        device=DEVICE,
        resolution_weights=resolution_weights,
        positional_weight=POSITIONAL_WEIGHT,
        use_amp=True,
    )
    
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(
            path=OUTPUT_DIR / 'best_model.pt',
            epoch=epoch,
            model=model,
            head=head,
            optimizer=optimizer,
            val_loss=val_loss,
            track_names=track_names,
            modality='atac',
            resolutions=RESOLUTIONS,
        )
        print(f"  -> Saved best model (val_loss={val_loss:.4f})")

print(f"\nTraining complete! Best val_loss: {best_val_loss:.4f}")
print(f"Checkpoint saved to: {OUTPUT_DIR / 'best_model.pt'}")

## 9. Inference

Load the finetuned model and run inference on a test region.

In [ ]:
import pyfaidx

# Load checkpoint
checkpoint = torch.load(OUTPUT_DIR / 'best_model.pt', map_location=DEVICE)
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
print(f"Validation loss: {checkpoint['val_loss']:.4f}")
print(f"Track names: {checkpoint['track_names']}")

In [ ]:
# Define a test region
TEST_CHROM = 'chr21'
TEST_CENTER = 46125988

# Compute region bounds
half_len = SEQUENCE_LENGTH // 2
test_start = TEST_CENTER - half_len
test_end = TEST_CENTER + half_len

print(f"Test region: {TEST_CHROM}:{test_start:,}-{test_end:,}")

In [ ]:
# Get one-hot encoded sequence
def sequence_to_onehot(sequence: str) -> np.ndarray:
    seq_bytes = np.frombuffer(sequence.encode('ascii'), dtype=np.uint8)
    onehot = np.zeros((len(seq_bytes), 4), dtype=np.uint8)
    mapping = np.full(128, -1, dtype=np.int8)
    for i, char in enumerate(['A', 'C', 'G', 'T']):
        mapping[ord(char)] = i
        mapping[ord(char.lower())] = i
    # clip(0, 127) prevents crash on non-ascii if they exist
    indices = mapping[seq_bytes.clip(0, 127)]
    mask = indices >= 0
    onehot[np.where(mask)[0], indices[mask]] = 1
    return onehot

# Fetch sequence from genome
fasta = pyfaidx.Fasta(GENOME_FASTA)
    sequence_str = str(fasta[TEST_CHROM][test_start:test_end])

sequence = sequence_to_onehot(sequence_str)
sequence = torch.from_numpy(sequence).unsqueeze(0).to(DEVICE)  # (1, seq_len, 4)

print(f"Sequence shape: {sequence.shape}")

In [ ]:
# Run inference
model.eval()
head.eval()

with torch.no_grad():
    organism_idx = torch.zeros(1, dtype=torch.long, device=DEVICE)
    
    # Forward through trunk
    outputs = model(sequence, organism_idx, return_embeddings=True)
    
    # Get embeddings for each resolution
    embeddings_dict = {}
    for res in RESOLUTIONS:
        emb_key = f'embeddings_{res}bp'
        if emb_key in outputs:
            embeddings_dict[res] = outputs[emb_key]
    
    # Forward through head
    predictions = head(embeddings_dict, organism_idx)

# Print prediction shapes
for res, pred in predictions.items():
    print(f"Resolution {res}bp: {pred.shape}")

## 10. Visualization

Compare predictions to ground truth BigWig signal.

In [ ]:
import matplotlib.pyplot as plt
import pyBigWig

# Load ground truth from first BigWig
with pyBigWig.open(BIGWIG_FILES[0]) as bw:
    ground_truth = bw.values(TEST_CHROM, test_start, test_end)
    ground_truth = np.nan_to_num(np.array(ground_truth), nan=0.0)

# Get 1bp predictions for first track
pred_1bp = predictions[1][0, :, 0].cpu().numpy()  # (seq_len,)

# Create position array
positions = np.arange(test_start, test_end)

# Plot comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Plot a zoomed window (10kb around center)
window = 5000
center_idx = SEQUENCE_LENGTH // 2
slice_range = slice(center_idx - window, center_idx + window)

axes[0].plot(positions[slice_range], ground_truth[slice_range], 'b-', linewidth=0.5, alpha=0.8)
axes[0].set_ylabel('Ground Truth')
axes[0].set_title(f'ATAC-seq Signal: {TEST_CHROM}:{test_start + center_idx - window:,}-{test_start + center_idx + window:,}')

axes[1].plot(positions[slice_range], pred_1bp[slice_range], 'r-', linewidth=0.5, alpha=0.8)
axes[1].set_ylabel('Prediction')
axes[1].set_xlabel('Genomic Position')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'prediction_comparison.png', dpi=150)
plt.show()

print(f"Saved plot to {OUTPUT_DIR / 'prediction_comparison.png'}")

In [ ]:
# Compute correlation
from scipy.stats import pearsonr, spearmanr

# Use non-zero positions for correlation (genomic signals are sparse)
mask = ground_truth > 0
if mask.sum() > 10:
    pearson_r, _ = pearsonr(ground_truth[mask], pred_1bp[mask])
    spearman_r, _ = spearmanr(ground_truth[mask], pred_1bp[mask])
    print(f"Pearson r (non-zero positions): {pearson_r:.4f}")
    print(f"Spearman r (non-zero positions): {spearman_r:.4f}")
else:
    print("Not enough non-zero positions for correlation")

## Next Steps

- **LoRA finetuning**: LoRA adapters instead of linear probing. See `scripts/finetune_atac_bigwigs.py` for an example.
- **Multi-task finetuning**: Use tracks from multiple modalities in a finetuning run.